# Notebook 03 — Joint Training
**Phase 2-3:** Freeze decoder, add LSTM head, train with joint loss.
L = MSE(reconstruction) + lambda * CrossEntropy(classification)

## 0. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import json
import numpy as np
import matplotlib.pyplot as plt
import torch

from src.config import (
    LAMBDA, JOINT_EPOCHS, MODELS_DIR, PLOTS_DIR, OUTPUTS_DIR, SEED, CLASSES,
)
from src.model import Encoder, Decoder, LSTMHead, TemporalPoolHead, JointModel, freeze_decoder
from src.dataset import compute_class_weights
from src.train import joint_train, plot_joint_loss, joint_train_lambda_sweep

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

CLIPS_DIR = OUTPUTS_DIR / "clips_packed"
print(f"PyTorch: {torch.__version__}  Device: {DEVICE}")


PyTorch: 2.13.0+cu130  Device: cuda


## 1. Load Data

In [2]:
meta_file = CLIPS_DIR / "meta.json"
with open(meta_file) as f:
    meta = json.load(f)

train_npy = meta["train"]["npy_paths"]
y_train   = np.array(meta["train"]["y"])
val_npy   = meta["val"]["npy_paths"]
y_val     = np.array(meta["val"]["y"])
test_npy  = meta["test"]["npy_paths"]
y_test    = np.array(meta["test"]["y"])

class_weights = compute_class_weights(y_train)
print(f"Train : {len(train_npy):,} clips  {dict(zip(CLASSES, np.bincount(y_train)))}")
print(f"Val   : {len(val_npy):,} clips    {dict(zip(CLASSES, np.bincount(y_val)))}")
print(f"Test  : {len(test_npy):,} clips   {dict(zip(CLASSES, np.bincount(y_test)))}")
print(f"Class weights: {dict(zip(CLASSES, class_weights.round(3)))}")


Train : 53,587 clips  {'Normal': np.int64(46929), 'Fighting': np.int64(1375), 'Robbery': np.int64(2016), 'Arson': np.int64(1436), 'Burglary': np.int64(1831)}
Val   : 13,273 clips    {'Normal': np.int64(11947), 'Fighting': np.int64(149), 'Robbery': np.int64(510), 'Arson': np.int64(71), 'Burglary': np.int64(596)}
Test  : 9,416 clips   {'Normal': np.int64(7898), 'Fighting': np.int64(146), 'Robbery': np.int64(97), 'Arson': np.int64(337), 'Burglary': np.int64(938)}
Class weights: {'Normal': np.float32(0.228), 'Fighting': np.float32(7.794), 'Robbery': np.float32(5.316), 'Arson': np.float32(7.463), 'Burglary': np.float32(5.853)}


## 2. Load Pre-trained Weights and Freeze Decoder

In [3]:
encoder = Encoder()
decoder = Decoder()
encoder.load_state_dict(torch.load(MODELS_DIR / "encoder_pretrained.pth", map_location=DEVICE))
decoder.load_state_dict(torch.load(MODELS_DIR / "decoder_pretrained.pth", map_location=DEVICE))

freeze_decoder(decoder)

trainable = sum(p.numel() for p in decoder.parameters() if p.requires_grad)
total     = sum(p.numel() for p in decoder.parameters())
print(f"Decoder trainable: {trainable}  frozen: {total - trainable}")

[INFO] Decoder frozen (8 parameter tensors).
Decoder trainable: 0  frozen: 2270819


## 3. Build Joint Model

In [4]:
lstm_head   = LSTMHead()
joint_model = JointModel(encoder, decoder, lstm_head)

encoder_params = sum(p.numel() for p in joint_model.encoder.parameters())
head_params    = sum(p.numel() for p in joint_model.temporal_head.parameters())
frozen_total   = sum(p.numel() for p in joint_model.decoder.parameters())
print(f"Encoder params (frozen during joint train): {encoder_params:,}")
print(f"LSTMHead params (trainable)               : {head_params:,}")
print(f"Decoder params (frozen)                   : {frozen_total:,}")


Encoder params (frozen during joint train): 2,191,520
LSTMHead params (trainable)               : 206,213
Decoder params (frozen)                   : 2,270,819


## 4. Joint Training (Lambda = 0.5)

In [ ]:
history = joint_train(
    joint_model, train_npy, y_train, val_npy, y_val,
    lam=LAMBDA, device=DEVICE,
    class_weights=class_weights,
    use_packed=True,
)


## 5. Loss and Accuracy Curves

In [ ]:
plot_joint_loss(history, save_path=PLOTS_DIR / "joint_loss.png")
plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(history["train_acc"], label="Train Accuracy", linewidth=1.5)
ax.plot(history["val_acc"],   label="Val Accuracy",   linewidth=1.5, linestyle="--")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.set_title("Classification Accuracy During Joint Training", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "joint_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

[OK] Plot saved: /home/mjl/softwarica/ANN/outputs/plots/joint_loss.png


/tmp/ipykernel_4010/1791857050.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Lambda Sensitivity Analysis

In [ ]:
print("\nRunning lambda sensitivity analysis (trains 3 models)...")
lambda_results = joint_train_lambda_sweep(
    train_npy, y_train, val_npy, y_val,
    lambdas=(0.1, 0.5, 1.0), device=DEVICE,
    class_weights=class_weights, use_packed=True,
)

lambdas    = list(lambda_results.keys())
val_losses = [lambda_results[l]["val_loss"] for l in lambdas]
val_accs   = [lambda_results[l]["val_acc"]  for l in lambdas]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))
ax1.bar([str(l) for l in lambdas], val_losses, color="#4472C4")
ax1.set_title("Val Loss by Lambda", fontweight="bold")
ax1.set_xlabel("Lambda"); ax1.set_ylabel("Validation Loss")
ax2.bar([str(l) for l in lambdas], val_accs, color="#ED7D31")
ax2.set_title("Val Accuracy by Lambda", fontweight="bold")
ax2.set_xlabel("Lambda"); ax2.set_ylabel("Validation Accuracy")
plt.suptitle("Lambda Sensitivity Analysis", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "lambda_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Summary

In [ ]:
joint_model.load_state_dict(
    torch.load(MODELS_DIR / "joint_model_best.pth", map_location=DEVICE))
print(f"Best val accuracy: {max(history['val_acc']):.4f}")
print(f"Best val loss    : {min(history['val_loss']):.4f}")
print("\n[DONE] Joint model saved: joint_model_best.pth")